In [ ]:
!python -m pip install xarray[complete] cdsapi cfgrib eccodes


In [ ]:
!pip install tensorflow matplotlib seaborn

In [ ]:
!pip show tensorflow

Name: tensorflow
Version: 2.19.0
Summary: TensorFlow is an open source machine learning framework for everyone.
Home-page: https://www.tensorflow.org/
Author: Google Inc.
Author-email: packages@tensorflow.org
License: Apache 2.0
Location: /usr/local/lib/python3.12/dist-packages
Requires: absl-py, astunparse, flatbuffers, gast, google-pasta, grpcio, h5py, keras, libclang, ml-dtypes, numpy, opt-einsum, packaging, protobuf, requests, setuptools, six, tensorboard, termcolor, typing-extensions, wrapt
Required-by: dopamine_rl, tensorflow-text, tensorflow_decision_forests, tf_keras


In [ ]:
!ls {GOOGLE_DRIVE_DATA_DIR}/src

__pycache__  score.py


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:

import os
from pathlib import Path

In [3]:

# Depending on your combination of package versions, this can raise a lot of TF warnings...
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr
import tensorflow as tf
import tensorflow.keras as keras
from tensorflow.keras.layers import *
import tensorflow.keras.backend as K
import seaborn as sns
import pickle
# from {GOOGLE_DRIVE_DATA_DIR}/src.score import *
from collections import OrderedDict



In [4]:
GOOGLE_DRIVE_DATA_DIR = "/content/drive/MyDrive/WeatherGo/data"
# for 7 lead days
# GOOGLE_DRIVE_VARS_FNAMES = list(map(lambda x: x.stem + x.suffix, Path(GOOGLE_DRIVE_DATA_DIR).glob("*.nc")))

In [ ]:
"""
1 day: 24 hours:
data for training: 24 hours + 24 hours
2 day: 2*24 hours:
data for training: 2*24 hours + 2*24
...
7 day: 7*24 hours:
data for training: 7*24+7*24
"""

In [ ]:
# how to run jupyter notebook on wsl:
# https://harshityadav95.medium.com/jupyter-notebook-in-windows-subsystem-for-linux-wsl-8b46fdf0a536

In [ ]:
# playground

In [25]:
var_dict = {v: None for v in train_ds.dtypes.keys()}

NameError: name 'train_ds' is not defined

In [ ]:
var_dict

{'valid_time': None, 'z': None, 't': None}

In [ ]:
# ds['valid_time'].data

In [ ]:
data = []

In [ ]:
generic_level = xr.DataArray([1], coords={'level': [1]}, dims=['level'])

In [ ]:
# for var, _ in var_dict.items():
#     try:
#         print(var)
#         data.append(train_ds[var].expand_dims({'level': generic_level}, 1))
#     except err:
#         ...

for var, levels in var_dict.items():
    try:
        data.append(train_ds[var].sel(level=levels))
    except Exception:
        data.append(train_ds[var].expand_dims({'level': generic_level}, 1))


In [ ]:
data = xr.concat(data, 'level').transpose('time', 'latitude', 'longitude', 'level')

In [ ]:
data.shape

(168, 113, 41, 3)

In [ ]:
lead_time = 24

In [ ]:
init_time = data.isel(time=slice(None, -lead_time)).time

In [ ]:
init_time

<xarray.DataArray 'time' (time: 0)> Size: 0B
array([], dtype=int64)
Coordinates:
  * time     (time) int64 0B 
Attributes:
    long_name:      initial time of forecast
    standard_name:  forecast_reference_time
    units:          seconds since 1970-01-01
    calendar:       proleptic_gregorian

In [ ]:
valid_time = data.isel(time=slice(lead_time, None)).time

In [ ]:
valid_time

<xarray.DataArray 'time' (time: 0)> Size: 0B
array([], dtype=int64)
Coordinates:
  * time     (time) int64 0B 
Attributes:
    long_name:      initial time of forecast
    standard_name:  forecast_reference_time
    units:          seconds since 1970-01-01
    calendar:       proleptic_gregorian

In [ ]:
n_samples = data.isel(time=slice(0, -lead_time)).shape[0]
n_samples

0

In [ ]:
batch_size = 2


In [ ]:
import numpy as np

In [ ]:
idxs = np.arange(n_samples)
np.random.shuffle(idxs)
idxs

array([], dtype=int64)

In [ ]:
i = 0
idxs = idxs[i * batch_size:(i + 1) * batch_size]
X = data.isel(time=idxs).values
y = data.isel(time=idxs+lead_time).values


In [ ]:
# playground

In [5]:
class DataGenerator(keras.utils.Sequence):
    def __init__(self, ds, var_dict, lead_time, batch_size=2, shuffle=True, load=True, mean=None, std=None):
        """
        Data generator for WeatherBench data.
        Template from https://stanford.edu/~shervine/blog/keras-how-to-generate-data-on-the-fly
        Args:
            ds: Dataset containing all variables
            var_dict: Dictionary of the form {'var': level}. Use None for level if data is of single level
            lead_time: Lead time in hours
            batch_size: Batch size
            shuffle: bool. If True, data is shuffled.
            load: bool. If True, datadet is loaded into RAM.
            mean: If None, compute mean from data.
            std: If None, compute standard deviation from data.
        """
        self.ds = ds
        self.var_dict = var_dict
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.lead_time = lead_time

        data = []
        generic_level = xr.DataArray([1], coords={'level': [1]}, dims=['level'])
        for var, levels in var_dict.items():
            try:
                data.append(ds[var].sel(level=levels))
            except Exception:
                data.append(ds[var].expand_dims({'level': generic_level}, 1))

        self.data = xr.concat(data, 'level').transpose('time', 'latitude', 'longitude', 'level')
        self.mean = self.data.mean(('time', 'latitude', 'longitude')).compute() if mean is None else mean
        self.std = self.data.std('time').mean(('latitude', 'longitude')).compute() if std is None else std
        # Normalize
        self.data = (self.data - self.mean) / self.std
        self.n_samples = self.data.isel(time=slice(0, -lead_time)).shape[0]
        self.init_time = self.data.isel(time=slice(None, -lead_time)).time
        self.valid_time = self.data.isel(time=slice(lead_time, None)).time

        self.on_epoch_end()

        # For some weird reason calling .load() earlier messes up the mean and std computations
        if load: print('Loading data into RAM'); self.data.load()

    def __len__(self):
        'Denotes the number of batches per epoch'
        return int(np.ceil(self.n_samples / self.batch_size))

    def __getitem__(self, i):
        'Generate one batch of data'
        idxs = self.idxs[i * self.batch_size:(i + 1) * self.batch_size]
        X = self.data.isel(time=idxs).values
        y = self.data.isel(time=idxs + self.lead_time).values
        return X, y

    def on_epoch_end(self):
        'Updates indexes after each epoch'
        self.idxs = np.arange(self.n_samples)
        if self.shuffle == True:
            np.random.shuffle(self.idxs)

## architecture

In [6]:
class PeriodicPadding2D(tf.keras.layers.Layer):
    def __init__(self, pad_width, **kwargs):
        super().__init__(**kwargs)
        self.pad_width = pad_width

    def call(self, inputs, **kwargs):
        if self.pad_width == 0:
            return inputs
        inputs_padded = tf.concat(
            [inputs[:, :, -self.pad_width:, :], inputs, inputs[:, :, :self.pad_width, :]], axis=2)
        # Zero padding in the lat direction
        inputs_padded = tf.pad(inputs_padded, [[0, 0], [self.pad_width, self.pad_width], [0, 0], [0, 0]])
        return inputs_padded

    def get_config(self):
        config = super().get_config()
        config.update({'pad_width': self.pad_width})
        return config


class PeriodicConv2D(tf.keras.layers.Layer):
    def __init__(self, filters,
                 kernel_size,
                 conv_kwargs={},
                 **kwargs, ):
        super().__init__(**kwargs)
        self.filters = filters
        self.kernel_size = kernel_size
        self.conv_kwargs = conv_kwargs
        if type(kernel_size) is not int:
            assert kernel_size[0] == kernel_size[1], 'PeriodicConv2D only works for square kernels'
            kernel_size = kernel_size[0]
        pad_width = (kernel_size - 1) // 2
        self.padding = PeriodicPadding2D(pad_width)
        self.conv = Conv2D(
            filters, kernel_size, padding='valid', **conv_kwargs
        )

    def call(self, inputs):
        return self.conv(self.padding(inputs))

    def get_config(self):
        config = super().get_config()
        config.update({'filters': self.filters, 'kernel_size': self.kernel_size, 'conv_kwargs': self.conv_kwargs})
        return config

In [7]:
def build_cnn(filters, kernels, input_shape, dr=0):
    """Fully convolutional network"""
    x = input = tf.keras.Input(shape=input_shape)
    for f, k in zip(filters[:-1], kernels[:-1]):
        x = PeriodicConv2D(f, k)(x)
        x = tf.keras.layers.LeakyReLU()(x)
        if dr > 0: x = tf.keras.layers.Dropout(dr)(x)
    output = PeriodicConv2D(filters[-1], kernels[-1])(x)
    return keras.models.Model(input, output)

## train and run predictions

In [8]:
# years period of training
# CAUTION: do not use when dataset contains less than year data covered (in days, i believe), in other words leave as downloaded, do not slice for years period
# train_start_year = 2025
# train_end_year = 2025
# val_start_year = 2025
# val_end_year = 2025
# test_start_year = 2025
# test_end_year = 2025

# how far (in hours) we going to make prediction
lead_times = [24*num_days for num_days in range(1, 7+1)]
batch_size = 2


In [9]:
def load_data(lead_time):
  files = os.listdir(Path(f'{GOOGLE_DRIVE_DATA_DIR}/{lead_time}'))
  dss = [xr.open_dataset(f'{GOOGLE_DRIVE_DATA_DIR}/{lead_time}/{f}', decode_cf=False) for f in files]

  ds = xr.concat(dss, 'level').transpose('time', 'latitude', 'longitude', 'level')
  # ds_train = ds.sel(time=slice(train_start_year, train_end_year))
  # ds_test = ds.sel(time=slice(test_start_year, test_end_year))
  ds_train = ds
  ds_test = ds
  return ds_train, ds_test



In [10]:
# for each lead time, in hours: [1*24, ..., 7*24] create and train model
models = {lead_time: None for lead_time in lead_times}
data = {lead_time: load_data(lead_time) for lead_time in lead_times}
generators = {lead_time: [] for lead_time in lead_times}
time_grid = 24 # lead time in hours

for lead_hour in data.keys():
    ds_train = data[lead_hour][0].drop_vars(["step", 'number', 'isobaricInhPa', 'valid_time'])
    ds_test = data[lead_hour][1].drop_vars(["step", 'number', 'isobaricInhPa', 'valid_time'])
    data[lead_hour] = (ds_train, ds_test)

dic = OrderedDict({var: None for var in data[24][0].dtypes.keys()}) # anny day. any dataset contains the same variables

n_lat, n_lon =  data[24][0].sizes['latitude'], data[24][0].sizes['longitude']
n_vars = len(dic)
input_shape = (n_lat, n_lon, n_vars)



In [ ]:
for lead_time in lead_times:
  print(f"training for lead time: {lead_time}")
  dg_train = DataGenerator(data[lead_time][0], dic, lead_time, batch_size=batch_size, load=True)
  dg_valid = DataGenerator(data[lead_time][1], dic, lead_time, batch_size=batch_size, mean=dg_train.mean, std=dg_train.std, shuffle=False)
  generators[lead_time] = [dg_train, dg_valid]

  cnn = build_cnn([64, 64, 64, 64, n_vars], [5, 5, 5, 5, 5], input_shape)
  cnn.compile(keras.optimizers.Adam(1e-4), 'mse')
  cnn.fit(dg_train, epochs=1, validation_data=dg_valid,
          callbacks=[tf.keras.callbacks.EarlyStopping(
                        monitor='val_loss',
                        min_delta=0,
                        patience=2,
                        verbose=1,
                        mode='auto'
                    )]
         )
  models[lead_time] = cnn


In [ ]:
# playground

In [ ]:
cnn = build_cnn([64, 64, 64, 64, 2], [5, 5, 5, 5, 5], input_shape)

In [ ]:
cnn.compile(keras.optimizers.Adam(1e-4), 'mse')


In [ ]:
cnn.summary()


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 113, 41, 2)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ periodic_conv2d                 │ (None, 113, 41, 64)    │         3,264 │
│ (PeriodicConv2D)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu (LeakyReLU)         │ (None, 113, 41, 64)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ periodic_conv2d_1               │ (None, 113, 41, 64)    │       102,464 │
│ (PeriodicConv2D)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_1 (LeakyReLU)       │ (None, 113, 41, 64)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ periodic_conv2d_2               │ (None, 113, 41, 64)    │       102,464 │
│ (PeriodicConv2D)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_2 (LeakyReLU)       │ (None, 113, 41, 64)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ periodic_conv2d_3               │ (None, 113, 41, 64)    │       102,464 │
│ (PeriodicConv2D)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_3 (LeakyReLU)       │ (None, 113, 41, 64)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ periodic_conv2d_4               │ (None, 113, 41, 2)     │         3,202 │
│ (PeriodicConv2D)                │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 313,858 (1.20 MB)

 Trainable params: 313,858 (1.20 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
cnn.fit(dg_train, epochs=1, validation_data=dg_valid,
          callbacks=[tf.keras.callbacks.EarlyStopping(
                        monitor='val_loss',
                        min_delta=0,
                        patience=2,
                        verbose=1,
                        mode='auto'
                    )]
         )

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


72/72 ━━━━━━━━━━━━━━━━━━━━ 7s 32ms/step - loss: 1.6968 - val_loss: 0.6501


In [ ]:
# playground

## create predictions

In [ ]:
def create_predictions(model, dg):
    """Create predictions for non-iterative model"""
    preds = model.predict(dg)
    # Unnormalize
    preds = preds * dg.std.values + dg.mean.values
    fcs = []
    lev_idx = 0
    for var, levels in dg.var_dict.items():
        if levels is None:
            fcs.append(xr.DataArray(
                preds[:, :, :, lev_idx],
                dims=['time', 'latitude', 'longitude'],
                coords={'time': dg.valid_time, 'latitude': dg.ds.latitude, 'longitude': dg.ds.longitude},
                name=var
            ))
            lev_idx += 1
        else:
            nlevs = len(levels)
            fcs.append(xr.DataArray(
                preds[:, :, :, lev_idx:lev_idx+nlevs],
                dims=['time', 'latitude', 'longitude', 'level'],
                coords={'time': dg.valid_time, 'latitude': dg.ds.latitude, 'longitude': dg.ds.longitude, 'level': levels},
                name=var
            ))
            lev_idx += nlevs
    return xr.merge(fcs)

In [ ]:
fc = create_predictions(cnn, dg_valid)


72/72 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step


In [ ]:
fc

<xarray.Dataset> Size: 5MB
Dimensions:    (time: 144, latitude: 113, longitude: 41)
Coordinates:
  * time       (time) int64 1kB 1762041600 1762045200 ... 1762552800 1762556400
  * latitude   (latitude) float64 904B 59.0 58.75 58.5 58.25 ... 31.5 31.25 31.0
  * longitude  (longitude) float64 328B 49.0 49.25 49.5 ... 58.5 58.75 59.0
Data variables:
    z          (time, latitude, longitude) float32 3MB 1.436e+04 ... 1.497e+04
    t          (time, latitude, longitude) float32 3MB 271.0 271.2 ... 286.6

fc

In [ ]:
#  save models
directory_path = f"{GOOGLE_DRIVE_DATA_DIR}/baselines"
os.makedirs(directory_path, exist_ok=True)
for lead_time in range(1, 6+1): # last model lacked data to train on
  models[lead_time*time_grid].save_weights(f'{directory_path}/{lead_time*time_grid}_hours_cnn.weights.h5')

In [ ]:
# test predict

days_n = 1
lead_time = days_n * 24
dg_val = generators[lead_time][1]
fc_24_hours = create_predictions(models[lead_time], dg_val)
fc_24_hours.sel(time=np.datetime64("2025-11-02T04:00").astype("int64"), method="nearest")


72/72 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step


<xarray.Dataset> Size: 242kB
Dimensions:    (latitude: 113, longitude: 41)
Coordinates:
  * latitude   (latitude) float64 904B 59.0 58.75 58.5 58.25 ... 31.5 31.25 31.0
  * longitude  (longitude) float64 328B 49.0 49.25 49.5 ... 58.5 58.75 59.0
    time       int64 8B 1762041600
Data variables: (12/13)
    t          (latitude, longitude) float32 19kB 270.6 270.9 ... 284.2 285.3
    z          (latitude, longitude) float32 19kB 1.468e+04 ... 1.51e+04
    o3         (latitude, longitude) float32 19kB 8.684e-08 ... 8.572e-08
    ciwc       (latitude, longitude) float32 19kB 2.027e-06 ... 2.66e-07
    r          (latitude, longitude) float32 19kB 63.26 65.6 65.26 ... 34.1 33.6
    pv         (latitude, longitude) float32 19kB 5.055e-07 ... 6.229e-07
    ...         ...
    crwc       (latitude, longitude) float32 19kB 3.373e-07 ... 2.428e-07
    q          (latitude, longitude) float32 19kB 0.002121 0.002127 ... 0.002038
    clwc       (latitude, longitude) float32 19kB 5.411e-05 ... 1.312e-05
    vo         (latitude, longitude) float32 19kB 2.327e-06 ... -8.281e-06
    w          (latitude, longitude) float32 19kB -0.001641 0.0135 ... 0.01736
    v          (latitude, longitude) float32 19kB -2.385 -2.945 ... -1.426

In [ ]:
time_lead = 6*24
checkpoint = f'{directory_path}/{time_lead}_hours_cnn.weights.h5'
loaded_model = build_cnn([64, 64, 64, 64, n_vars], [5, 5, 5, 5, 5], input_shape)
loaded_model.load_weights(checkpoint)

create_predictions(loaded_model, generators[time_lead][1])


12/12 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step


<xarray.Dataset> Size: 6MB
Dimensions:    (time: 24, latitude: 113, longitude: 41)
Coordinates:
  * time       (time) int64 192B 1762473600 1762477200 ... 1762552800 1762556400
  * latitude   (latitude) float64 904B 59.0 58.75 58.5 58.25 ... 31.5 31.25 31.0
  * longitude  (longitude) float64 328B 49.0 49.25 49.5 ... 58.5 58.75 59.0
Data variables: (12/13)
    t          (time, latitude, longitude) float32 445kB 273.0 273.1 ... 280.0
    z          (time, latitude, longitude) float32 445kB 1.44e+04 ... 1.486e+04
    o3         (time, latitude, longitude) float32 445kB 8.907e-08 ... 8.91e-08
    ciwc       (time, latitude, longitude) float32 445kB 3.973e-06 ... 3.499e-07
    r          (time, latitude, longitude) float32 445kB 79.73 80.29 ... 33.83
    pv         (time, latitude, longitude) float32 445kB 8.456e-08 ... 8.372e-07
    ...         ...
    crwc       (time, latitude, longitude) float32 445kB 6.117e-07 ... 3.238e-07
    q          (time, latitude, longitude) float32 445kB 0.003679 ... 0.002762
    clwc       (time, latitude, longitude) float32 445kB 0.0001209 ... 1.334e-06
    vo         (time, latitude, longitude) float32 445kB 1.688e-06 ... -4.786...
    w          (time, latitude, longitude) float32 445kB 0.03926 ... 0.01464
    v          (time, latitude, longitude) float32 445kB -7.431 ... 0.8181

In [ ]:
# Load weights from externally trained model
# for up to 6 days only
loaded_models = dict()
for lead_time in range(1, 6+1):
  cnn = build_cnn([64, 64, 64, 64, n_vars], [5, 5, 5, 5, 5], input_shape)
  checkpoint = f'{directory_path}/{lead_time*time_grid}_hours_cnn.weights.h5'
  cnn.load_weights(checkpoint)

  loaded_models[lead_time*time_grid] = cnn


In [ ]:
fc_24_hours.sel(time=np.datetime64("2025-11-02T04:00").astype("int64"), method="nearest")

<xarray.Dataset> Size: 242kB
Dimensions:    (latitude: 113, longitude: 41)
Coordinates:
  * latitude   (latitude) float64 904B 59.0 58.75 58.5 58.25 ... 31.5 31.25 31.0
  * longitude  (longitude) float64 328B 49.0 49.25 49.5 ... 58.5 58.75 59.0
    time       int64 8B 1762041600
Data variables: (12/13)
    t          (latitude, longitude) float32 19kB 270.6 270.9 ... 284.2 285.3
    z          (latitude, longitude) float32 19kB 1.468e+04 ... 1.51e+04
    o3         (latitude, longitude) float32 19kB 8.684e-08 ... 8.572e-08
    ciwc       (latitude, longitude) float32 19kB 2.027e-06 ... 2.66e-07
    r          (latitude, longitude) float32 19kB 63.26 65.6 65.26 ... 34.1 33.6
    pv         (latitude, longitude) float32 19kB 5.055e-07 ... 6.229e-07
    ...         ...
    crwc       (latitude, longitude) float32 19kB 3.373e-07 ... 2.428e-07
    q          (latitude, longitude) float32 19kB 0.002121 0.002127 ... 0.002038
    clwc       (latitude, longitude) float32 19kB 5.411e-05 ... 1.312e-05
    vo         (latitude, longitude) float32 19kB 2.327e-06 ... -8.281e-06
    w          (latitude, longitude) float32 19kB -0.001641 0.0135 ... 0.01736
    v          (latitude, longitude) float32 19kB -2.385 -2.945 ... -1.426

In [ ]:
def from_Kelvin_to_Celsius(t):
  return t - 273.15

In [ ]:
predictions = dict()
for lead_time in lead_times:
  dg_val = generators[lead_time][1]
  fc = create_predictions(models[lead_time], dg_val)
  predictions[lead_time] = fc


72/72 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step
48/48 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step


AttributeError: 'NoneType' object has no attribute 'predict'

In [ ]:
# for now, up to 6 days inclusively:(
predictions[6*24]

<xarray.Dataset> Size: 6MB
Dimensions:    (time: 24, latitude: 113, longitude: 41)
Coordinates:
  * time       (time) int64 192B 1762473600 1762477200 ... 1762552800 1762556400
  * latitude   (latitude) float64 904B 59.0 58.75 58.5 58.25 ... 31.5 31.25 31.0
  * longitude  (longitude) float64 328B 49.0 49.25 49.5 ... 58.5 58.75 59.0
Data variables: (12/13)
    t          (time, latitude, longitude) float32 445kB 273.0 273.1 ... 280.0
    z          (time, latitude, longitude) float32 445kB 1.44e+04 ... 1.486e+04
    o3         (time, latitude, longitude) float32 445kB 8.907e-08 ... 8.91e-08
    ciwc       (time, latitude, longitude) float32 445kB 3.973e-06 ... 3.499e-07
    r          (time, latitude, longitude) float32 445kB 79.73 80.29 ... 33.83
    pv         (time, latitude, longitude) float32 445kB 8.456e-08 ... 8.372e-07
    ...         ...
    crwc       (time, latitude, longitude) float32 445kB 6.117e-07 ... 3.238e-07
    q          (time, latitude, longitude) float32 445kB 0.003679 ... 0.002762
    clwc       (time, latitude, longitude) float32 445kB 0.0001209 ... 1.334e-06
    vo         (time, latitude, longitude) float32 445kB 1.688e-06 ... -4.786...
    w          (time, latitude, longitude) float32 445kB 0.03926 ... 0.01464
    v          (time, latitude, longitude) float32 445kB -7.431 ... 0.8181

In [ ]:
def get_forecast(ts: np.datetime64, lon, lat, variable: str = "temperature", temp_unit: str ="C", lead_hours=24):
  # ts = np.datetime64("2025-11-02T04:00")
  # lon, lat: specific points at map, TODO: region cover
  """
  variable: ["t", ]
  """
  vars_map = {
      "temperature": "t",
      "geopotential": "z",
      "ozone_mass_mixing_ratio": "o3",
      "specific_cloud_ice_water_content": "ciwc",
      "relative_humidity": "r",
      "potential_vorticity": "pv",
      "u_component_of_wind": "u",
      "specific_rain_water_content": "u",
      "specific_humidity": "crwc",
      "specific_cloud_liquid_water_content": "q",
      "vorticity": "vo",
      "vertical_velocity": "w",
      "v_component_of_wind": "v"
  }

  prediction = predictions[lead_hours].sel(time=ts.astype("int64"),
         latitude=lat, longitude=lon,
         method="nearest")[vars_map.get(variable, "t")].item()

  if variable == "temperature":
    if temp_unit == "C":
      prediction = from_Kelvin_to_Celsius(prediction)

  return prediction


In [ ]:
import numpy as np

In [ ]:
test_data = [
    {'city': 'Bila Tserkva', 'lon': 49.7968, 'lat': 30.1278},
    {'city': 'Krzvzi Rih', 'lon': 47.9105, 'lat': 33.3918},
    {'city': 'Mukachevo', 'lon': 48.4493, 'lat': 22.7117},
  ]

In [ ]:
get_forecast(ts=np.datetime64("2025-11-02T04:00"),
             lon=test_data[0]['lon'],
             lat=test_data[0]['lat'],
             variable="v_component_of_wind",
             lead_hours=24)

-2.148341178894043

269.5394287109375